### **Step 1 — Install and import libraries**

In [1]:
import os
import librosa
import numpy as np
from tqdm import tqdm


### **Step 2 — Define paths**

In [2]:
BASE = r"D:\Deepfake_Audio_Project"

CLEAN = os.path.join(BASE, "clean_data")
FEATURES = os.path.join(BASE, "features")
SPECTRO = os.path.join(FEATURES, "spectrograms")
MFCCS = os.path.join(FEATURES, "mfcc")
# create folders if they don't exist

### **Step 3 — Create feature folders**


In [3]:
for split in ["train","dev","eval"]:
    for cls in ["real","fake"]:
        os.makedirs(os.path.join(SPECTRO, split, cls), exist_ok=True)
        os.makedirs(os.path.join(MFCCS, split, cls), exist_ok=True)


## **PART A — Spectrogram Generation (CNN Input)**

### **Step 4 — Function to create Mel Spectrogram**

In [4]:
def save_spectrogram_matrix(audio_path, save_path):

    # 1. Load audio
    y, sr = librosa.load(audio_path, sr=16000)

    # 2. Normalize length (important for CNN)
    max_len = 4 * sr   # 4 seconds

    if len(y) < max_len:
        y = np.pad(y, (0, max_len - len(y)))
    else:
        y = y[:max_len]

    # 3. Create mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=128,
        fmax=8000
    )

    # 4. Convert to log scale
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # 5. Normalize values (VERY important for CNN stability)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

    # 6. Save as numpy matrix
    np.save(save_path, mel_db.astype(np.float32))


### **Step 5 — Generate spectrogram dataset**

In [5]:
def process_split(split):

    for cls in ["real","fake"]:
        input_folder = os.path.join(CLEAN, split, cls)
        output_folder = os.path.join(SPECTRO, split, cls)

        for file in tqdm(os.listdir(input_folder)):
            audio_file = os.path.join(input_folder, file)

            save_file = os.path.join(
                output_folder,
                file.replace(".flac",".npy")
            )

            save_spectrogram_matrix(audio_file, save_file)


In [6]:
process_split("train")

  0%|          | 0/2580 [00:00<?, ?it/s]

C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 22800/22800 [04:29<00:00, 84.59it/s]


In [7]:
process_split("dev")

  0%|          | 0/2548 [00:00<?, ?it/s]

100%|██████████| 22296/22296 [06:09<00:00, 60.33it/s]


In [8]:
process_split("eval")

100%|██████████| 63882/63882 [17:31<00:00, 60.77it/s]


## **PART B — MFCC Extraction (Boosting Input)**

### **Step 6 — Extract MFCC Features**

In [9]:
def extract_mfcc(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)

    return np.mean(mfcc.T, axis=0)


### **Step 7 — Save MFCC arrays**

In [10]:
def create_mfcc_dataset(split):

    X = []
    y = []

    for label,cls in enumerate(["real","fake"]):

        folder = os.path.join(CLEAN, split, cls)

        for file in tqdm(os.listdir(folder)):
            path = os.path.join(folder, file)

            feat = extract_mfcc(path)

            X.append(feat)
            y.append(label)

    np.save(os.path.join(MFCCS, f"X_{split}.npy"), np.array(X))
    np.save(os.path.join(MFCCS, f"y_{split}.npy"), np.array(y))


In [11]:
create_mfcc_dataset("train")

  0%|          | 0/2580 [00:00<?, ?it/s]

100%|██████████| 22800/22800 [05:37<00:00, 67.56it/s]


In [12]:
create_mfcc_dataset("dev")

  0%|          | 0/2548 [00:00<?, ?it/s]

100%|██████████| 22296/22296 [05:34<00:00, 66.69it/s]


In [13]:
create_mfcc_dataset("eval")

100%|██████████| 63882/63882 [15:27<00:00, 68.91it/s] 
